# CTU-Chat Remote Reranker on Colab

Run every cell in order on a GPU runtime. The notebook loads `BAAI/bge-reranker-v2-m3`, serves batched scores through FastAPI, and exposes the service through a temporary Cloudflare Quick Tunnel. Use it for experiments, not permanent production hosting.

In [1]:
!pip -q install -U FlagEmbedding fastapi uvicorn httpx
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.5 MB/s eta 0:00:00
cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


In [2]:
import getpass
import secrets
import threading
import torch
from FlagEmbedding import FlagReranker

MODEL_NAME = 'BAAI/bge-reranker-v2-m3'
MAX_DOCUMENTS = 64
MAX_TEXT_CHARS = 8000
API_KEY = getpass.getpass('Create an API key for this Colab session: ').strip()
if len(API_KEY) < 16:
    raise ValueError('Use an API key with at least 16 characters')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (switch Colab runtime to GPU)')
reranker = FlagReranker(MODEL_NAME, use_fp16=torch.cuda.is_available())
inference_lock = threading.Lock()
print('Model ready:', MODEL_NAME)

GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Model ready: BAAI/bge-reranker-v2-m3


In [3]:
from typing import Annotated
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title='CTU Remote Reranker', version='1.0')

class Candidate(BaseModel):
    id: str = Field(min_length=1, max_length=128)
    text: str = Field(max_length=MAX_TEXT_CHARS)

class RerankRequest(BaseModel):
    query: str = Field(min_length=1, max_length=4000)
    documents: list[Candidate] = Field(min_length=1, max_length=MAX_DOCUMENTS)

def authorize(authorization: str | None) -> None:
    expected = f'Bearer {API_KEY}'
    if authorization is None or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='Unauthorized')

@app.get('/health')
def health(authorization: Annotated[str | None, Header()] = None):
    authorize(authorization)
    return {
        'status': 'ok',
        'model': MODEL_NAME,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'max_documents': MAX_DOCUMENTS,
    }

@app.post('/rerank')
def rerank(request: RerankRequest, authorization: Annotated[str | None, Header()] = None):
    authorize(authorization)
    ids = [document.id for document in request.documents]
    if len(ids) != len(set(ids)):
        raise HTTPException(status_code=422, detail='Document IDs must be unique')
    pairs = [[request.query, document.text] for document in request.documents]
    with inference_lock:
        raw_scores = reranker.compute_score(pairs, normalize=False)
    if hasattr(raw_scores, 'tolist'):
        raw_scores = raw_scores.tolist()
    if not isinstance(raw_scores, (list, tuple)):
        raw_scores = [raw_scores]
    if len(raw_scores) != len(request.documents):
        raise HTTPException(status_code=500, detail='Model returned an invalid score count')
    return {
        'model': MODEL_NAME,
        'scores': [
            {'id': document.id, 'score': float(score)}
            for document, score in zip(request.documents, raw_scores)
        ],
    }

In [4]:
import time
import uvicorn
import httpx

def run_api():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='info')

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
headers = {'Authorization': f'Bearer {API_KEY}'}
for _ in range(60):
    try:
        response = httpx.get('http://127.0.0.1:8000/health', headers=headers, timeout=2)
        response.raise_for_status()
        print(response.json())
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('FastAPI did not start')

INFO:     Started server process [476]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:39816 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'model': 'BAAI/bge-reranker-v2-m3', 'device': 'Tesla T4', 'max_documents': 64}


In [5]:
import pathlib
import re
import subprocess

tunnel_log_path = pathlib.Path('/tmp/ctu-cloudflared.log')
tunnel_log_handle = tunnel_log_path.open('w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)
PUBLIC_URL = None
for _ in range(120):
    time.sleep(0.5)
    log_text = tunnel_log_path.read_text(errors='replace') if tunnel_log_path.exists() else ''
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break
if PUBLIC_URL is None:
    raise RuntimeError('Cloudflare tunnel did not publish a URL. Log:\n' + log_text[-2000:])
print('Public URL:', PUBLIC_URL)
print('\nPut these values in your local .env:')
print('RAG_RERANKER_BACKEND=remote')
print(f'RAG_REMOTE_RERANKER_URL={PUBLIC_URL}')
print('RAG_REMOTE_RERANKER_API_KEY=<the same key entered above>')

Public URL: https://comfortable-places-guam-announce.trycloudflare.com

Put these values in your local .env:
RAG_RERANKER_BACKEND=remote
RAG_REMOTE_RERANKER_URL=https://comfortable-places-guam-announce.trycloudflare.com
RAG_REMOTE_RERANKER_API_KEY=<the same key entered above>


In [6]:
payload = {
    'query': 'Lệ phí xin cấp bản sao văn bằng tốt nghiệp là bao nhiêu?',
    'documents': [
        {'id': 'diploma', 'text': 'Phiếu đề nghị cấp bản sao văn bằng. Lệ phí cấp một bản sao là 10.000 đồng.'},
        {'id': 'tuition', 'text': 'Bảng mức thu học phí chương trình đại học chính quy khóa 52.'},
    ],
}
response = httpx.post(f'{PUBLIC_URL}/rerank', headers=headers, json=payload, timeout=60)
response.raise_for_status()
print(response.json())

ConnectError: [Errno -2] Name or service not known

INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 422 Unprocessable Entity
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 422 Unprocessable Entity
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 422 Unprocessable Entity
INFO:     14.161.190.207:0 - "GET /health HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     14.161.190.207:0 - "POST /rerank HTTP/1.